## Behandlung der Variable: Analyse der Zeitabstände (`time_gap`)

In diesem Abschnitt widmen wir uns der zentralen Zielvariable unseres Modells: den Zeitabständen der Fahrer zum Etappensieger (`time_gap`). Im Radsport ist die Modellierung dieser Variable eine besondere Herausforderung, da sie zwei fundamentale Eigenschaften aufweist:

1. **Reine Rechtsschiefe (Extreme Skewness):** Der Etappensieger hat einen Abstand von `0` Sekunden. Ein Großteil des Hauptfeldes (Peloton) kommt oft mit demselben oder einem geringen Zeitabstand im Ziel an. Danach reißt das Feld ab, und es bildet sich ein langer "Schwanz" (Long Tail) von Fahrern mit massiven Zeitabständen (z. B. das *Gruppetto* bei Bergetappen).
2. **Ausreißer vs. sportliche Realität:** Sehr große Abweichungen (Stundencharakter bei schweren Rundfahrten) sind statistisch gesehen extreme Ausreißer, spiegeln jedoch die reale Dynamik des Rennens wider.

### Ziel dieser Analyse:
* **Verteilungsprüfung:** Mathematische Bestimmung der Schiefe und Wölbung mittels `scipy.stats`.
* **Umgang mit NaNs / DNF:** Definition von Strategien für Fahrer, die das Ziel nicht erreicht haben (Did Not Finish).
* **Transformationen:** Evaluierung von Methoden (z. B. Log-Transformation oder Box-Cox), um die Rechtsschiefe für Lukas' XGBRanker oder Regressionsmodelle zu normalisieren.

In [1]:
import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

In [2]:
# DF Laden
path = "../../data/processed/11_cleaned_master_data.pkl"

df = pd.read_pickle(path)

In [3]:
print(f"Shape: {df.shape[0]:,} Zeilen x {df.shape[1]} Spalten")

Shape: 222,818 Zeilen x 48 Spalten


In [8]:
# 1. Basis-Statistik
print("--- Analyse der Zeitabstände ---")
print(df['time_gap'].describe())

# 2. NaNs und Nuller zählen
print(f"\nFehlende Werte (NaNs): {df['time_gap'].isna().sum():,}")


# 3. Schreibweise und Datentyp prüfen
print(df['time_gap'].dtype)
print()
print(df['time_gap'].head(20))

--- Analyse der Zeitabstände ---
count     222818
unique      6747
top       ,,0:00
freq       39510
Name: time_gap, dtype: object

Fehlende Werte (NaNs): 0
object

0     20:5120:51
1       0:020:02
2       0:530:53
3       0:570:57
4       0:590:59
5       1:021:02
6         ,,1:02
7       1:041:04
8       1:051:05
9       1:061:06
10      1:071:07
11      1:081:08
12      1:121:12
13      1:131:13
14        ,,1:13
15      1:161:16
16      1:181:18
17      1:241:24
18      1:251:25
19      1:261:26
Name: time_gap, dtype: object


# Aufbau der Spalte

- Siegerzeit ganz oben "4:30:074:30:07"
- danach Gap zum Vordermann "0:070:07"
- wenn zu gleicher Zeit mit einem anderen ",,0:07"
- zudem Sonderwertungen wie "DNF", "DNS", "DSQ", "OTL", "HD":
    {
      "rank": "DNF",
      "rider_url": "david-lopez",
      "time_gap": "n/a"
    }

```json
{
  "race": "giro-d-italia",
  "year": 2009,
  "url": "https://www.procyclingstats.com/race/giro-d-italia/2009/stage-20",
  "metadata": {
    "departure": "Napoli",
    "arrival": "Anagni",
    "distance": "203 km",
    "vertical_meters": "1577",
    "profile_score": "53",
    "won_how": "Sprint à deux",
    "avg_speed": "45.092 km/h",
    "race_ranking": "n/a"
  },
  "results": [
    {
      "rank": "1",
      "rider_url": "philippe-gilbert",
      "time_gap": "4:30:074:30:07"
    },
    {
      "rank": "2",
      "rider_url": "thomas-voeckler",
      "time_gap": "0:020:02"
    },
    {
      "rank": "3",
      "rider_url": "stefano-garzelli",
      "time_gap": "0:070:07"
    },
    {
      "rank": "4",
      "rider_url": "allan-davis",
      "time_gap": ",,0:07"
    },
    {
      "rank": "5",
      "rider_url": "sebastien-hinault",
      "time_gap": ",,0:07"
    },
    {
      "rank": "6",
      "rider_url": "franco-pellizotti",
      "time_gap": ",,0:07"
    },
    {
      "rank": "7",
      "rider_url": "edvald-boasson-hagen",
      "time_gap": ",,0:07"
    },
    {
      "rank": "8",
      "rider_url": "giovanni-visconti",
      "time_gap": ",,0:07"
    },
    {
      "rank": "9",
      "rider_url": "simon-gerrans",
      "time_gap": ",,0:07"
    },
    {
      "rank": "10",
      "rider_url": "serge-pauwels",
      "time_gap": ",,0:07"
    },
    {
      "rank": "11",
      "rider_url": "denis-menchov",
      "time_gap": ",,0:07"
    },
    {
      "rank": "12",
      "rider_url": "danilo-di-luca",
      "time_gap": ",,0:07"
    },
    {
      "rank": "13",
      "rider_url": "rubens-bertogliati",
      "time_gap": ",,0:07"
    },
    {
      "rank": "14",
      "rider_url": "ruggero-marzoli",
      "time_gap": ",,0:07"
    },
{..................................}
    {
      "rank": "141",
      "rider_url": "bradley-wiggins",
      "time_gap": ",,7:55"
    },
    {
      "rank": "142",
      "rider_url": "olivier-kaisen",
      "time_gap": "8:248:24"
    },
    {
      "rank": "143",
      "rider_url": "jure-golcer",
      "time_gap": "8:448:44"
    },
    {
      "rank": "144",
      "rider_url": "francesco-de-bonis",
      "time_gap": "13:2813:28"
    },
    {
      "rank": "145",
      "rider_url": "giuseppe-palumbo",
      "time_gap": ",,13:28"
    },
{..................................}
    {
      "rank": "169",
      "rider_url": "matthieu-sprick",
      "time_gap": ",,13:28"
    },
    {
      "rank": "DNF",
      "rider_url": "david-lopez",
      "time_gap": "n/a"
    }
  ]
}
```

In [13]:
# Alle Sonderformen wie DNF, DNS... idendifizieren:

# Alle einzigartigen Ränge extrahieren und sortieren
all_unique_ranks = sorted(df['rank'].dropna().unique().astype(str))

print(f"Gesamtanzahl einzigartiger Werte in 'rank': {len(all_unique_ranks)}")
print("\n--- Alle einzigartigen Werte im Überblick ---")
print(all_unique_ranks)

Gesamtanzahl einzigartiger Werte in 'rank': 207

--- Alle einzigartigen Werte im Überblick ---
['1.0', '10.0', '100.0', '101.0', '102.0', '103.0', '104.0', '105.0', '106.0', '107.0', '108.0', '109.0', '11.0', '110.0', '111.0', '112.0', '113.0', '114.0', '115.0', '116.0', '117.0', '118.0', '119.0', '12.0', '120.0', '121.0', '122.0', '123.0', '124.0', '125.0', '126.0', '127.0', '128.0', '129.0', '13.0', '130.0', '131.0', '132.0', '133.0', '134.0', '135.0', '136.0', '137.0', '138.0', '139.0', '14.0', '140.0', '141.0', '142.0', '143.0', '144.0', '145.0', '146.0', '147.0', '148.0', '149.0', '15.0', '150.0', '151.0', '152.0', '153.0', '154.0', '155.0', '156.0', '157.0', '158.0', '159.0', '16.0', '160.0', '161.0', '162.0', '163.0', '164.0', '165.0', '166.0', '167.0', '168.0', '169.0', '17.0', '170.0', '171.0', '172.0', '173.0', '174.0', '175.0', '176.0', '177.0', '178.0', '179.0', '18.0', '180.0', '181.0', '182.0', '183.0', '184.0', '185.0', '186.0', '187.0', '188.0', '189.0', '19.0', '190.0'

In [ ]:
import os
import pandas as pd

# Basis-Ordner, in dem deine verarbeiteten Pickles liegen
folder_path = "../../data/processed/"

print("--- Analyse der 'rank'-Werte über alle Pipeline-Schritte ---\n")

# Schleife von 01 bis 11 (formatiert mit führender Null)
for i in range(1, 12):
    prefix = f"{i:02d}"

    # Wir suchen nach einer Datei, die mit der aktuellen Nummer beginnt und auf .pkl endet
    found_file = None
    if os.path.exists(folder_path):
        for file in os.listdir(folder_path):
            if file.startswith(prefix) and file.endswith(".pkl"):
                found_file = file
                break

    # Falls keine passende Datei gefunden wurde, skippen wir
    if not found_file:
        print(f"Skipping: Keine .pkl-Datei für Schritt {prefix} gefunden.")
        continue

    # Datei laden
    file_full_path = os.path.join(folder_path, found_file)
    try:
        current_df = pd.read_pickle(file_full_path)

        print(f"📄 Datei gefunden: {found_file}")
        print(f"   Shape: {current_df.shape[0]:,} Zeilen")

        if 'rank' in current_df.columns:
            # Alle einzigartigen Ränge als Strings extrahieren (ohne NaNs für die Übersicht)
            distinct_ranks = current_df['rank'].dropna().unique()

            # Sortieren, falls möglich (Sonderzeichen/Kombinationen fangen wir ab)
            try:
                distinct_ranks = sorted(distinct_ranks.astype(str))
            except:
                distinct_ranks = list(distinct_ranks.astype(str))

            print(f"   Echte Datentypen in 'rank': {current_df['rank'].apply(type).unique()}")
            print(f"   Anzahl NaNs in 'rank': {current_df['rank'].isna().sum():,}")
            print(f"   Distinct Values (erste 15): {distinct_ranks[:15]}")
            if len(distinct_ranks) > 15:
                print(f"   ... und {len(distinct_ranks) - 15} weitere Werte.")
        else:
            print("   ⚠️ Spalte 'rank' existiert in diesem DataFrame nicht!")

        print("-" * 60)

    except Exception as e:
        print(f"❌ Fehler beim Laden von {found_file}: {e}")
        print("-" * 60)

--- Analyse der 'rank'-Werte über alle Pipeline-Schritte ---

📄 Datei gefunden: 01_cleaned_master_data.pkl
   Shape: 225,692 Zeilen
   Echte Datentypen in 'rank': [<class 'str'>]
   Anzahl NaNs in 'rank': 0
   Distinct Values (erste 15): ['1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111']
   ... und 198 weitere Werte.
------------------------------------------------------------
Skipping: Keine .pkl-Datei für Schritt 02 gefunden.
Skipping: Keine .pkl-Datei für Schritt 03 gefunden.
📄 Datei gefunden: 04_cleaned_master_data.pkl
   Shape: 225,692 Zeilen
   Echte Datentypen in 'rank': [<class 'str'>]
   Anzahl NaNs in 'rank': 0
   Distinct Values (erste 15): [np.str_('1'), np.str_('10'), np.str_('100'), np.str_('101'), np.str_('102'), np.str_('103'), np.str_('104'), np.str_('105'), np.str_('106'), np.str_('107'), np.str_('108'), np.str_('109'), np.str_('11'), np.str_('110'), np.str_('111')]
   ... und 198 weitere Werte.
----------------------

In [ ]:
import os
import pandas as pd

folder_path = "../../data/processed/"
# Wir suchen gezielt nach der Datei, die mit '08' beginnt
file_08 = None
for file in os.listdir(folder_path):
    if file.startswith("08") and file.endswith(".pkl"):
        file_08 = file
        break

if file_08:
    path_08 = os.path.join(folder_path, file_08)
    df_08 = pd.read_pickle(path_08)

    print(f"📄 Datei geladen: {file_08}")
    print(f"Gesamtzeilen im Datensatz: {len(df_08):,}")
    print(f"Anzahl fehlender Werte (NaNs) in 'rank': {df_08['rank'].isna().sum():,}\n")

    # Alle unique Werte extrahieren, zu Strings machen und sortieren
    # Wir nutzen eine Python-Liste für die vollständige Ausgabe
    all_ranks_08 = sorted(df_08['rank'].dropna().unique().astype(str))

    print(f"--- Alle {len(all_ranks_08)} distinct Values für 'rank' in Schritt 08 ---")
    print(all_ranks_08)
else:
    print("❌ Keine passende .pkl-Datei für Schritt 08 im Ordner gefunden!")

📄 Datei geladen: 08_cleaned_master_data.pkl
Gesamtzeilen im Datensatz: 225,692
Anzahl fehlender Werte (NaNs) in 'rank': 0

--- Alle 213 distinct Values für 'rank' in Schritt 08 ---
[np.str_('1'), np.str_('10'), np.str_('100'), np.str_('101'), np.str_('102'), np.str_('103'), np.str_('104'), np.str_('105'), np.str_('106'), np.str_('107'), np.str_('108'), np.str_('109'), np.str_('11'), np.str_('110'), np.str_('111'), np.str_('112'), np.str_('113'), np.str_('114'), np.str_('115'), np.str_('116'), np.str_('117'), np.str_('118'), np.str_('119'), np.str_('12'), np.str_('120'), np.str_('121'), np.str_('122'), np.str_('123'), np.str_('124'), np.str_('125'), np.str_('126'), np.str_('127'), np.str_('128'), np.str_('129'), np.str_('13'), np.str_('130'), np.str_('131'), np.str_('132'), np.str_('133'), np.str_('134'), np.str_('135'), np.str_('136'), np.str_('137'), np.str_('138'), np.str_('139'), np.str_('14'), np.str_('140'), np.str_('141'), np.str_('142'), np.str_('143'), np.str_('144'), np.str_('